# Ancient Ambient Sound Level Comparison

This notebook compares two methods for estimating "ancient ambient" underwater noise levels:
- **Method 1 (ship-filtered)**: Excludes time windows with ship presence, using AIS-derived ship metrics
- **Method 2 (unfiltered)**: Uses rolling percentile statistics over longer windows without ship filtering

**Ship data analysis date range**: 2026-02-07 to 2026-02-13

> **Environment requirement**: This notebook requires the `orcasound` conda environment.
> Activate it before launching Jupyter: `conda activate orcasound`

In [ ]:
import sys
import datetime
from datetime import datetime, timedelta
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, '../src')
from orcasound_noise.analysis.partitioned_accessor import PartitionedAccessor
from orcasound_noise.utils.hydrophone import Hydrophone

In [ ]:
# Analysis parameters
HYDROPHONE = Hydrophone.ORCASOUND_LAB
ANALYSIS_END = datetime(2026, 2, 13, 23, 59, 59)
SHIP_DATA_START = datetime(2026, 2, 7)
EXTENDED_START = datetime(2026, 1, 31)
AWS_PROFILE = "ambient-sound-team"
AWS_REGION = "us-west-2"
CONFIDENCE_THRESHOLD = 0.5
COMM_BAND = (500, 15000)
PERCENTILES = [0.05, 0.10, 0.25, 0.50]
METHOD1_WINDOWS = [1, 2, 5, 7]   # days, capped by ship data availability
METHOD2_WINDOWS = [1, 2, 3, 5, 7, 10, 14]  # days, no ship data required
S3_SHIP_METRICS = "s3://acoustic-sandbox/ambient-sound-analysis/temp_ship_metrics"